# Optical Spatial Dispersion of Conductivity plotting

#### モジュール

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import c, epsilon_0, h, hbar, e
import re
import os
import ipynbname
NB_NAME = ipynbname.name()

#### データセットの定義

In [ ]:
# 公開用サンプル

files_material_comparison = {
    "title": "Material comparison (example)",
    "files": {
        "./../materials/MaterialA/NC/SOC/wann96/wannierberri_OSD/MaterialA-SDCT_asym_all_iter-0000.dat": "MaterialA",
        "./../materials/MaterialB/NC/SOC/wann96/wannierberri_OSD/MaterialB-SDCT_asym_all_iter-0000.dat": "MaterialB",
    },
}

files_kmesh_convergence = {
    "title": "MaterialA: kmesh convergence (example)",
    "files": {
        "./../materials/MaterialA/NC/SOC/wann96/wannierberri_OSD/MaterialA_eta0.05_NK30-SDCT_asym_all_iter-0000.dat": "(0.05, 30x30x30)",
        "./../materials/MaterialA/NC/SOC/wann96/wannierberri_OSD/MaterialA_eta0.05_NK40-SDCT_asym_all_iter-0000.dat": "(0.05, 40x40x40)",
        "./../materials/MaterialA/NC/SOC/wann96/wannierberri_OSD/MaterialA_eta0.05_NK50-SDCT_asym_all_iter-0000.dat": "(0.05, 50x50x50)",
    },
}

files_eta_convergence = {
    "title": "MaterialA: smearing convergence (example)",
    "files": {
        "./../materials/MaterialA/NC/SOC/wann96/wannierberri_OSD/MaterialA_eta0.1_NK50-SDCT_asym_all_iter-0000.dat": "(0.100, 50x50x50)",
        "./../materials/MaterialA/NC/SOC/wann96/wannierberri_OSD/MaterialA_eta0.05_NK50-SDCT_asym_all_iter-0000.dat": "(0.050, 50x50x50)",
    },
}

#### データ変換関数

In [ ]:
# 3階テンソルから自然光学活性への変換

def calc_rho_theta(filename: str):

    # 複素三階テンソルへの成型
    tmp = np.loadtxt(filename, unpack=True)
    energy = tmp[1,:]                             # eV
    omega =  energy * e / hbar                    # 1/s
    sigma_real = tmp[2:56:2,:]                    # (Siemens/m) * m = Siemens
    sigma_imag = tmp[3:56:2,:]
    sigma_cmpx = sigma_real + 1j * sigma_imag
    sigma = sigma_cmpx.reshape(3,3,3,-1)

    # 反対称部分の抽出
    eta_abc = sigma / omega / epsilon_0           # Siemens / (1/s * Farad/m) = m
    eta_bac = np.swapaxes(eta_abc, 0, 1)
    eta_as = 0.5 * (eta_abc - eta_bac)

    # レヴィチビタ記号
    epsilon = np.zeros((3, 3, 3), dtype=int)
    epsilon[0, 1, 2] = 1.0
    epsilon[1, 2, 0] = 1.0
    epsilon[2, 0, 1] = 1.0
    epsilon[0, 2, 1] = -1.0
    epsilon[2, 1, 0] = -1.0
    epsilon[1, 0, 2] = -1.0

    # アインシュタインの縮約記法
    gyro = 0.5 * np.einsum('ijk, jklm->ilm', epsilon, eta_as)
    gyro *= (360 / (2.0 * np.pi))                 # Radians to degrees
    gyro *= (1.0/1000.0)                          # Per meter to per milimeter

    # 旋光性
    rho_xx = 0.5 * omega**2 / c**2 * gyro[0,0,:].real
    rho_yy = 0.5 * omega**2 / c**2 * gyro[1,1,:].real
    rho_zz = 0.5 * omega**2 / c**2 * gyro[2,2,:].real
    rho_diag = (rho_xx + rho_yy + rho_zz) / 3

    # 円二色性
    theta_xx = 0.5 * omega**2 / c**2 * gyro[0,0,:].imag
    theta_yy = 0.5 * omega**2 / c**2 * gyro[1,1,:].imag
    theta_zz = 0.5 * omega**2 / c**2 * gyro[2,2,:].imag
    theta_diag = (theta_xx + theta_yy + theta_zz) / 3

    return energy, rho_xx, rho_yy, rho_zz, theta_zz, theta_yy, theta_zz


#### プロット

In [ ]:
plt.close("all")

# データセットの指定
dataset =  files_material_comparison
title = dataset["title"]
files = dataset["files"]

# サブプロットフレームの作成
fig, ax = plt.subplots(2, 3, figsize=(11, 6), sharex=True)
fig.canvas.header_visible = False

# エネルギーから波長への変換定数
hc_eV_nm = 1239.84  # h*c [eV・nm]

# 各サブプロットの描画設定
subplot_config = {
    (0, 0): {"title": r"$\rho_{xx}$ [mdeg]",   "xlim": (100, 1000), "ylim": (-4000, 4000)},
    (0, 1): {"title": r"$\rho_{yy}$ [mdeg]",   "xlim": (100, 1000), "ylim": (-4000, 4000)},
    (0, 2): {"title": r"$\rho_{zz}$ [mdeg]",   "xlim": (100, 1000), "ylim": (-4000, 4000)},
    (1, 0): {"title": r"$\theta_{xx}$ [mdeg]", "xlim": (100, 1000), "ylim": (-4000, 4000)},
    (1, 1): {"title": r"$\theta_{yy}$ [mdeg]", "xlim": (100, 1000), "ylim": (-4000, 4000)},
    (1, 2): {"title": r"$\theta_{zz}$ [mdeg]", "xlim": (100, 1000), "ylim": (-4000, 4000)},
}

# データの読み込みとプロット
for fname, label in files.items():
    energy, rho_xx, rho_yy, rho_zz, theta_xx, theta_yy, theta_zz = calc_rho_theta(fname)
    wavelength = hc_eV_nm / energy

    ax[0, 0].plot(wavelength, -2.0 * rho_xx, label=label)
    ax[0, 1].plot(wavelength, -2.0 * rho_yy, label=label)
    ax[0, 2].plot(wavelength, -2.0 * rho_zz, label=label)
    ax[1, 0].plot(wavelength, -2.0 * theta_xx, label=label)
    ax[1, 1].plot(wavelength, -2.0 * theta_yy, label=label)
    ax[1, 2].plot(wavelength, -2.0 * theta_zz, label=label)

# 描画設定の反映
for (i, j), cfg in subplot_config.items():
    if cfg.get("title") is not None:
        ax[i, j].set_title(cfg["title"])
    if cfg.get("xlim") is not None:
        ax[i, j].set_xlim(cfg["xlim"])
    if cfg.get("ylim") is not None:
        ax[i, j].set_ylim(cfg["ylim"])
    ax[i, j].set_xlabel("Wavelength [nm]")
    ax[i, j].tick_params(direction="in", which="both", top=True, right=True)
    ax[i,j].legend()

fig.suptitle(title, y=0.95, fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])

# 画像として保存
save_dir = f"./{NB_NAME}_save"
os.makedirs(save_dir, exist_ok=True)
fig.savefig(f"{save_dir}/{re.sub(r'[^\w\-]+', '_', title)}.png", dpi=300, bbox_inches="tight")